In [4]:
import pandas as pd
import sqlite3
from datetime import datetime, timedelta
import numpy as np

# Stap 1 E(Extract)

In [5]:
bron_conn = sqlite3.connect('../week_3/DE/sdm.db')


In [6]:
df_acc_inkoop = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop", bron_conn)
df_acc_inkoop_acc = pd.read_sql_query(
    "SELECT * FROM Accessoire_Inkoop_Accessoire", bron_conn)
df_acc_inkoop_lev = pd.read_sql_query(
    "SELECT * FROM Accessoire_Inkoop_Leverancier", bron_conn)

df_fiets_inkoop = pd.read_sql_query("SELECT * FROM Fiets_Inkoop", bron_conn)
df_fiets_inkoop_fab = pd.read_sql_query(
    "SELECT * FROM Fiets_Inkoop_Fabrikant", bron_conn)
df_fiets_inkoop_fiets = pd.read_sql_query(
    "SELECT * FROM Fiets_Inkoop_Fiets", bron_conn)

In [7]:
df_acc_verk = pd.read_sql_query("SELECT * FROM Accessoireverkoop", bron_conn)
df_acc_verk_acc = pd.read_sql_query(
    "SELECT * FROM Accessoireverkoop_Accessoire", bron_conn)
df_acc_verk_fil = pd.read_sql_query(
    "SELECT * FROM Accessoireverkoop_filiaal", bron_conn)
df_acc_verk_klant = pd.read_sql_query(
    "SELECT * FROM Accessoireverkoop_klant", bron_conn)
df_acc_verk_lev = pd.read_sql_query(
    "SELECT * FROM Accessoireverkoop_Leverancier", bron_conn)
df_acc_verk_mont = pd.read_sql_query(
    "SELECT * FROM Accessoireverkoop_Monteur", bron_conn)

df_fiets_verk = pd.read_sql_query("SELECT * FROM Fietsverkoop", bron_conn)
df_fiets_verk_fab = pd.read_sql_query(
    "SELECT * FROM Fietsverkoop_Fabrikant", bron_conn)
df_fiets_verk_fiets = pd.read_sql_query(
    "SELECT * FROM Fietsverkoop_Fiets", bron_conn)
df_fiets_verk_fil = pd.read_sql_query(
    "SELECT * FROM Fietsverkoop_filiaal", bron_conn)
df_fiets_verk_klant = pd.read_sql_query(
    "SELECT * FROM Fietsverkoop_klant", bron_conn)
df_fiets_verk_mont = pd.read_sql_query(
    "SELECT * FROM Fietsverkoop_Monteur", bron_conn)

In [8]:
df_ond = pd.read_sql_query("SELECT * FROM Onderhoud", bron_conn)
df_ond_fab = pd.read_sql_query("SELECT * FROM Onderhoud_Fabrikant", bron_conn)
df_ond_fiets = pd.read_sql_query("SELECT * FROM Onderhoud_Fiets", bron_conn)
df_ond_fil = pd.read_sql_query("SELECT * FROM Onderhoud_Filiaal", bron_conn)
df_ond_mont = pd.read_sql_query("SELECT * FROM Onderhoud_Monteur", bron_conn)

In [9]:
bron_conn.close()

# Stap 2 T (Transform)

In [10]:
dim_klant = pd.concat([df_acc_verk_klant, df_fiets_verk_klant]).drop_duplicates(
    subset=['klantnr']).reset_index(drop=True)
huidig_jaar = datetime.now().year
dim_klant['geboortedatum'] = pd.to_datetime(
    dim_klant['geboortedatum'], errors='coerce')
dim_klant['leeftijd'] = huidig_jaar - dim_klant['geboortedatum'].dt.year
dim_klant['leeftijdgroep'] = pd.cut(dim_klant['leeftijd'], bins=[
                                    0, 20, 40, 60, 100], labels=['<20', '20-40', '40-60', '60+'])
dim_klant = dim_klant.drop(columns=['leeftijd'])

In [11]:
dim_fiets_basis = pd.concat([df_fiets_inkoop_fiets, df_fiets_verk_fiets,
                            df_ond_fiets]).drop_duplicates(subset=['fietsnr'])
dim_fabrikant = pd.concat([df_fiets_inkoop_fab, df_fiets_verk_fab,
                          df_ond_fab]).drop_duplicates(subset=['fabrikantnr'])
# Koppel fiets aan fabrikant
dim_fiets = pd.merge(dim_fiets_basis, dim_fabrikant, left_on='fabrikant',
                     right_on='fabrikantnr', how='left', suffixes=('', '_fabrikant'))
dim_fiets = dim_fiets.rename(columns={
                             'naam': 'fabrikantnaam', 'adres': 'fabrikant_adres', 'plaats': 'fabrikant_plaats'})
dim_fiets['prijsklasse'] = np.where(
    dim_fiets['standaardprijs'] > 1000, 'Duur Segment', 'Basis Segment')

In [12]:
dim_acc_basis = pd.concat(
    [df_acc_inkoop_acc, df_acc_verk_acc]).drop_duplicates(subset=['accessoirenr'])
dim_lev = pd.concat([df_acc_inkoop_lev, df_acc_verk_lev]
                    ).drop_duplicates(subset=['leveranciernr'])
# Koppel accessoire aan leverancier
dim_accessoire = pd.merge(dim_acc_basis, dim_lev, left_on='leverancier',
                          right_on='leveranciernr', how='left', suffixes=('', '_leverancier'))
dim_accessoire = dim_accessoire.rename(
    columns={'naam_leverancier': 'leveranciernaam', 'adres': 'leverancier_adres', 'woonplaats': 'leverancier_woonplaats'})

In [13]:
dim_monteur_basis = pd.concat(
    [df_acc_verk_mont, df_fiets_verk_mont, df_ond_mont]).drop_duplicates(subset=['monteurnr'])
dim_filiaal = pd.concat([df_acc_verk_fil, df_fiets_verk_fil,
                        df_ond_fil]).drop_duplicates(subset=['filiaalnr'])
# Koppel monteur aan filiaal
dim_monteur = pd.merge(dim_monteur_basis, dim_filiaal, left_on='filiaal',
                       right_on='filiaalnr', how='left', suffixes=('', '_filiaal'))
dim_monteur = dim_monteur.rename(
    columns={'naam_filiaal': 'filiaalnaam', 'adres': 'filiaal_adres'})
dim_monteur['salarisgroep'] = np.where(
    dim_monteur['uurloon'] > 20, 'Senior', 'Medior')

In [14]:
dim_datum = pd.DataFrame(
    {'datum': pd.date_range(start='2020-01-01', end='2025-12-31')})
dim_datum['datum_id'] = dim_datum['datum'].dt.strftime('%Y%m%d').astype(int)
dim_datum['dag'] = dim_datum['datum'].dt.day
dim_datum['maand'] = dim_datum['datum'].dt.month
dim_datum['jaar'] = dim_datum['datum'].dt.year
dim_datum['maandnaam'] = dim_datum['datum'].dt.strftime('%B')
dim_datum['maand_id'] = dim_datum['datum'].dt.strftime('%Y%m').astype(int)

In [15]:
df_acc_inkoop_feit = df_acc_inkoop.copy()
df_acc_inkoop_feit = df_acc_inkoop_feit.rename(
    columns={'accessoire': 'accessoirenummer'})
df_acc_inkoop_feit['fietsnr'] = None  # Heeft geen fiets

df_fiets_inkoop_feit = df_fiets_inkoop.copy()
df_fiets_inkoop_feit = df_fiets_inkoop_feit.rename(
    columns={'fiets': 'fietsnr'})
df_fiets_inkoop_feit['accessoirenummer'] = None  # Heeft geen accessoire

feit_inkoop = pd.concat(
    [df_acc_inkoop_feit, df_fiets_inkoop_feit]).reset_index(drop=True)
feit_inkoop['inkoop_id'] = feit_inkoop.index + 1  # Nieuwe unieke DWH sleutel
# Maak maand_id: 2023 en 10 wordt 202310
feit_inkoop['maand_id'] = feit_inkoop['inkoopjaar'].astype(
    str) + feit_inkoop['inkoopmaand'].astype(str).str.zfill(2)
feit_inkoop['maand_id'] = feit_inkoop['maand_id'].astype(int)

In [16]:
feit_inkoop = pd.merge(
    feit_inkoop, dim_fiets[['fietsnr', 'inkoopprijs']], on='fietsnr', how='left')
feit_inkoop = pd.merge(feit_inkoop, dim_accessoire[['accessoirenr', 'inkoopprijs']],
                       left_on='accessoirenummer', right_on='accessoirenr', how='left', suffixes=('_fiets', '_acc'))
feit_inkoop['inkoopprijs_per_stuk'] = feit_inkoop['inkoopprijs_fiets'].fillna(
    feit_inkoop['inkoopprijs_acc'])
feit_inkoop['totale_inkoopkosten'] = feit_inkoop['aantal'] * \
    feit_inkoop['inkoopprijs_per_stuk']

feit_inkoop = feit_inkoop[['inkoop_id', 'maand_id', 'fietsnr',
                           'accessoirenummer', 'aantal', 'totale_inkoopkosten']]

In [17]:
df_acc_verk_feit = df_acc_verk.copy()
df_acc_verk_feit = df_acc_verk_feit.rename(
    columns={'accessoire': 'accessoirenummer', 'klant': 'klantnr', 'monteur': 'monteurnr'})
df_acc_verk_feit['fietsnr'] = None

df_fiets_verk_feit = df_fiets_verk.copy()
df_fiets_verk_feit = df_fiets_verk_feit.rename(
    columns={'fiets': 'fietsnr', 'klant': 'klantnr', 'monteur': 'monteurnr'})
df_fiets_verk_feit['accessoirenummer'] = None

feit_verkoop = pd.concat(
    [df_acc_verk_feit, df_fiets_verk_feit]).reset_index(drop=True)
feit_verkoop['verkoop_id'] = feit_verkoop.index + 1  # Unieke sleutel
feit_verkoop['datum'] = pd.to_datetime(feit_verkoop['datum'])
feit_verkoop['datum_id'] = feit_verkoop['datum'].dt.strftime(
    '%Y%m%d').astype(int)

In [18]:
feit_verkoop = pd.merge(
    feit_verkoop, dim_fiets[['fietsnr', 'inkoopprijs']], on='fietsnr', how='left')
feit_verkoop = pd.merge(feit_verkoop, dim_accessoire[['accessoirenr', 'inkoopprijs']],
                        left_on='accessoirenummer', right_on='accessoirenr', how='left', suffixes=('_fiets', '_acc'))
feit_verkoop['kostprijs_per_stuk'] = feit_verkoop['inkoopprijs_fiets'].fillna(
    feit_verkoop['inkoopprijs_acc'])
feit_verkoop['omzet'] = feit_verkoop['verkoopprijs']
feit_verkoop['winst'] = feit_verkoop['omzet'] - \
    (feit_verkoop['aantal'] * feit_verkoop['kostprijs_per_stuk'])

feit_verkoop = feit_verkoop[['verkoop_id', 'datum_id', 'klantnr', 'monteurnr',
                             'fietsnr', 'accessoirenummer', 'verkoopprijs', 'omzet', 'winst']]

In [19]:
feit_onderhoud = df_ond.copy()
feit_onderhoud = feit_onderhoud.rename(
    columns={'fiets': 'fietsnr', 'monteur': 'monteurnr'})
feit_onderhoud['datum'] = pd.to_datetime(feit_onderhoud['datum'])
feit_onderhoud['datum_id'] = feit_onderhoud['datum'].dt.strftime(
    '%Y%m%d').astype(int)

In [20]:
feit_onderhoud['start_dt'] = pd.to_datetime(
    feit_onderhoud['starttijd'].astype(str).str[:5], format='%H:%M', errors='coerce')
feit_onderhoud['eind_dt'] = pd.to_datetime(
    feit_onderhoud['eindtijd'].astype(str).str[:5], format='%H:%M', errors='coerce')
feit_onderhoud['duur'] = (feit_onderhoud['eind_dt'] -
                          feit_onderhoud['start_dt']).dt.total_seconds() / 3600.0

feit_onderhoud = feit_onderhoud[['onderhoudnr', 'datum_id',
                                 'fietsnr', 'monteurnr', 'starttijd', 'eindtijd', 'duur']]
feit_onderhoud = feit_onderhoud.rename(columns={'onderhoudnr': 'onderhoud_id'})

In [21]:
# Datum tabel opruimen
dim_datum = dim_datum.drop(columns=['datum'])

# Stap 3 L (Load)

In [22]:
dwh_conn = sqlite3.connect('fietsen_dwh.db')

dim_datum.to_sql('DIM_DATUM', dwh_conn, if_exists='replace', index=False)
dim_klant.to_sql('DIM_KLANT', dwh_conn, if_exists='replace', index=False)
dim_monteur.to_sql('DIM_MONTEUR', dwh_conn, if_exists='replace', index=False)
dim_fiets.to_sql('DIM_FIETS', dwh_conn, if_exists='replace', index=False)
dim_accessoire.to_sql('DIM_ACCESSOIRE', dwh_conn,
                      if_exists='replace', index=False)

feit_inkoop.to_sql('FEIT_INKOOP', dwh_conn, if_exists='replace', index=False)
feit_verkoop.to_sql('FEIT_VERKOOP', dwh_conn, if_exists='replace', index=False)
feit_onderhoud.to_sql('FEIT_ONDERHOUD', dwh_conn,
                      if_exists='replace', index=False)

dwh_conn.close()